# Week 1 — Urdu HMM POS Tagger
**Dataset**: Universal Dependencies Urdu Treebank (`ur_udtb`)  
**Goal**: Train transition + emission probabilities for a Hidden Markov Model (HMM) and save the model to `models/hmm_ud_urdu_week1.json`.

---

## Prerequisites
1. Install dependencies:
   ```bash
   pip install -r requirements.txt
   ```
2. Place the UD Urdu Treebank `.conllu` files in `data/ud/`:
   ```
   data/ud/ur_udtb-ud-train.conllu
   data/ud/ur_udtb-ud-dev.conllu
   data/ud/ur_udtb-ud-test.conllu
   ```
   Download from: https://github.com/UniversalDependencies/UD_Urdu-UDTB

## Outputs
- `models/hmm_ud_urdu_week1.json` — trained HMM (transition log-probs, emission log-probs, tag list, vocabulary).

---

## Workflow
1. **Section 1**: Setup — paths, imports
2. **Section 2**: Optional download helper
3. **Section 3**: Load & parse `.conllu` files
4. **Section 4**: Preprocess / normalize tokens
5. **Section 5**: Train HMM (transition + emission + Laplace smoothing)
6. **Section 6**: Save model to JSON
7. **Section 7**: Statistics & sanity checks

---
## Section 1 — Setup

In [ ]:
import os
import json
import math
from pathlib import Path
from collections import defaultdict, Counter

# ── Paths ─────────────────────────────────────────────────────────────
REPO_ROOT   = Path(os.getcwd()).parent          # one level up from notebooks/
DATA_DIR    = REPO_ROOT / "data" / "ud"
MODELS_DIR  = REPO_ROOT / "models"

TRAIN_FILE  = DATA_DIR / "ur_udtb-ud-train.conllu"
DEV_FILE    = DATA_DIR / "ur_udtb-ud-dev.conllu"
TEST_FILE   = DATA_DIR / "ur_udtb-ud-test.conllu"
MODEL_OUT   = MODELS_DIR / "hmm_ud_urdu_week1.json"

# ── HMM hyper-parameters ──────────────────────────────────────────────
START_TAG          = "<START>"
END_TAG            = "<END>"
UNK_TOKEN          = "<UNK>"
ALPHA_TRANSITION   = 1.0     # Laplace smoothing — transitions
ALPHA_EMISSION     = 1.0     # Laplace smoothing — emissions
MIN_WORD_COUNT     = 2       # words with count < 2 → replaced by UNK

MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir :", DATA_DIR)
print("Models dir:", MODELS_DIR)

---
## Section 2 — Optional download helper

Run this cell **only if** you have not yet placed the `.conllu` files in `data/ud/`.
It clones the UD Urdu-UDTB repository and copies the files into the expected locations.

In [ ]:
import subprocess
import shutil

def download_ud_urdu(data_dir: Path) -> None:
    """Clone UD_Urdu-UDTB into /tmp and copy .conllu files to data_dir."""
    tmp_clone = Path("/tmp/UD_Urdu-UDTB")
    if tmp_clone.exists():
        shutil.rmtree(tmp_clone)

    print("Cloning UD_Urdu-UDTB …")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/UniversalDependencies/UD_Urdu-UDTB.git",
         str(tmp_clone)],
        check=True
    )

    data_dir.mkdir(parents=True, exist_ok=True)
    for src in tmp_clone.glob("*.conllu"):
        dst = data_dir / src.name
        shutil.copy(src, dst)
        print(f"  Copied {src.name} → {dst}")
    print("Download complete.")


if not TRAIN_FILE.exists():
    print("Training file not found — attempting automatic download …")
    download_ud_urdu(DATA_DIR)
else:
    print("Training file already present:", TRAIN_FILE)

---
## Section 3 — Load & parse `.conllu` files

In [ ]:
def load_conllu(path: Path, tag_field: str = "UPOS"):
    """
    Parse a Universal Dependencies .conllu file.

    Returns
    -------
    list[list[tuple[str, str]]]
        Each outer list is a sentence; each inner tuple is (word_form, pos_tag).

    CoNLL-U columns (1-indexed):
      1=ID, 2=FORM, 3=LEMMA, 4=UPOS, 5=XPOS, 6=FEATS,
      7=HEAD, 8=DEPREL, 9=DEPS, 10=MISC
    """
    sentences = []
    current   = []
    tag_col   = 3 if tag_field.upper() == "UPOS" else 4   # 0-based

    with open(path, encoding="utf-8") as fh:
        for raw_line in fh:
            line = raw_line.rstrip("\n")

            # Blank line → sentence boundary
            if line == "":
                if current:
                    sentences.append(current)
                    current = []
                continue

            # Comment lines
            if line.startswith("#"):
                continue

            cols = line.split("\t")
            if len(cols) != 10:
                continue

            tok_id = cols[0]
            # Skip multiword (e.g. "1-2") and empty nodes (e.g. "1.1")
            if "-" in tok_id or "." in tok_id:
                continue

            word = cols[1]
            tag  = cols[tag_col]

            # Skip tokens with no annotation
            if word == "_" or tag == "_":
                continue

            current.append((word, tag))

    if current:
        sentences.append(current)

    return sentences


# --- Load all three splits -----------------------------------------------
train_sents = load_conllu(TRAIN_FILE)
dev_sents   = load_conllu(DEV_FILE)   if DEV_FILE.exists()  else []
test_sents  = load_conllu(TEST_FILE)  if TEST_FILE.exists() else []

print(f"Train sentences : {len(train_sents):>6}")
print(f"Dev   sentences : {len(dev_sents):>6}")
print(f"Test  sentences : {len(test_sents):>6}")
print()
print("First training sentence:")
print(train_sents[0] if train_sents else "(empty)")

---
## Section 4 — Preprocess / normalize tokens

In [ ]:
import unicodedata

def normalize_urdu(token: str) -> str:
    """
    Minimal normalization hook for Urdu tokens.

    Current operations
    ------------------
    • Strip surrounding whitespace.
    • NFC Unicode normalization (combines decomposed characters).

    Extend here in later weeks, e.g.:
    • Remove Arabic/Urdu diacritics (harakat): U+064B–U+065F
    • Normalize Yeh variants (U+06CC ↔ U+0649)
    • Normalize Heh variants (U+06C1 / U+06BE / U+0647)
    • Digit normalization (Arabic-Indic ↔ ASCII)
    """
    token = token.strip()
    token = unicodedata.normalize("NFC", token)
    return token


def preprocess(sentences):
    """Apply normalize_urdu to every word in every sentence."""
    result = []
    for sent in sentences:
        new_sent = [(normalize_urdu(w), t) for w, t in sent if w and t]
        if new_sent:
            result.append(new_sent)
    return result


train_sents = preprocess(train_sents)
dev_sents   = preprocess(dev_sents)
test_sents  = preprocess(test_sents)

print("Preprocessing complete.")
print("Example token after normalization:", train_sents[0][0] if train_sents else "n/a")

---
## Section 5 — Train HMM

### Algorithm
1. **Count** transition and emission frequencies from the training corpus.
2. **Apply Laplace (add-α) smoothing** to avoid zero probabilities.
3. **Store log-probabilities** to prevent underflow during decoding.

$$
P(\text{tag}_t \mid \text{tag}_{t-1}) = \frac{C(\text{tag}_{t-1}, \text{tag}_t) + \alpha}{C(\text{tag}_{t-1}) + \alpha \cdot |\text{tags}|}
$$

$$
P(\text{word} \mid \text{tag}) = \frac{C(\text{tag}, \text{word}) + \alpha}{C(\text{tag}) + \alpha \cdot |\text{vocab}|}
$$

In [ ]:
# ── Step 5a: Build vocabulary and replace rare words with UNK ────────────

word_freq = Counter(w for sent in train_sents for w, _ in sent)
tag_freq  = Counter(t for sent in train_sents for _, t in sent)

def replace_rare(sentences, word_freq, min_count, unk_token):
    result = []
    for sent in sentences:
        new_sent = [
            (w if word_freq.get(w, 0) >= min_count else unk_token, t)
            for w, t in sent
        ]
        result.append(new_sent)
    return result


train_sents_unk = replace_rare(train_sents, word_freq, MIN_WORD_COUNT, UNK_TOKEN)

# Recompute frequencies after UNK replacement
word_freq_unk = Counter(w for sent in train_sents_unk for w, _ in sent)
tag_freq_unk  = Counter(t for sent in train_sents_unk for _, t in sent)

TAGS  = sorted(tag_freq_unk.keys())
VOCAB = sorted(word_freq_unk.keys())

print(f"Unique POS tags : {len(TAGS)}")
print(f"Vocabulary size : {len(VOCAB)}  (after UNK replacement, min_count={MIN_WORD_COUNT})")
print(f"Tags            : {TAGS}")

In [ ]:
# ── Step 5b: Count transitions and emissions ─────────────────────────────

# transition_counts[prev_tag][curr_tag] = count
transition_counts = defaultdict(Counter)
# emission_counts[tag][word] = count
emission_counts   = defaultdict(Counter)

for sent in train_sents_unk:
    prev = START_TAG
    for word, tag in sent:
        transition_counts[prev][tag] += 1
        emission_counts[tag][word]   += 1
        prev = tag
    transition_counts[prev][END_TAG] += 1

print("Transition states with counts:", len(transition_counts))
print("Emission states  with counts:", len(emission_counts))

In [ ]:
# ── Step 5c: Compute smoothed log-probabilities ──────────────────────────

# Valid "next" states for transitions (never transition TO START)
next_states = TAGS + [END_TAG]
V_next      = len(next_states)
V_vocab     = len(VOCAB)

log_transition = {}   # log_transition[prev_tag][curr_tag]
log_emission   = {}   # log_emission[tag][word]

# ── Transitions ──────────────────────────────────────────────────────────
all_prev_states = [START_TAG] + TAGS   # START can precede; END cannot be prev
for prev in all_prev_states:
    log_transition[prev] = {}
    denom = sum(transition_counts[prev].values()) + ALPHA_TRANSITION * V_next
    for cur in next_states:
        num = transition_counts[prev][cur] + ALPHA_TRANSITION
        log_transition[prev][cur] = math.log(num / denom)

# ── Emissions ────────────────────────────────────────────────────────────
for tag in TAGS:
    log_emission[tag] = {}
    denom = sum(emission_counts[tag].values()) + ALPHA_EMISSION * V_vocab
    for word in VOCAB:
        num = emission_counts[tag][word] + ALPHA_EMISSION
        log_emission[tag][word] = math.log(num / denom)

print("log_transition computed for", len(log_transition), "prev-states")
print("log_emission   computed for", len(log_emission),   "tags")

---
## Section 6 — Save model to JSON

In [ ]:
model = {
    "meta": {
        "dataset"          : "UD Urdu Treebank (ur_udtb)",
        "tag_set"          : "UPOS",
        "train_sentences"  : len(train_sents),
        "train_tokens"     : sum(len(s) for s in train_sents),
        "vocabulary_size"  : len(VOCAB),
        "num_tags"         : len(TAGS),
        "alpha_transition" : ALPHA_TRANSITION,
        "alpha_emission"   : ALPHA_EMISSION,
        "min_word_count"   : MIN_WORD_COUNT,
    },
    "special_tokens": {
        "start": START_TAG,
        "end"  : END_TAG,
        "unk"  : UNK_TOKEN,
    },
    "tags" : TAGS,
    "vocab": VOCAB,
    "log_probs": {
        "transition": log_transition,
        "emission"  : log_emission,
    },
}

with open(MODEL_OUT, "w", encoding="utf-8") as fp:
    json.dump(model, fp, ensure_ascii=False, indent=2)

model_size_kb = MODEL_OUT.stat().st_size / 1024
print(f"Model saved to : {MODEL_OUT}")
print(f"File size      : {model_size_kb:.1f} KB")

---
## Section 7 — Statistics & sanity checks

In [ ]:
# ── Dataset statistics ───────────────────────────────────────────────────
print("═" * 50)
print(" DATASET STATISTICS")
print("═" * 50)

for split_name, sents in [("Train", train_sents), ("Dev", dev_sents), ("Test", test_sents)]:
    n_tok = sum(len(s) for s in sents)
    print(f"{split_name:>5}: {len(sents):>5} sentences, {n_tok:>7} tokens")

print()
print(f"Unique UPOS tags : {len(TAGS)}")
print(f"Tags             : {TAGS}")
print()
print(f"Vocabulary (after UNK, min_count={MIN_WORD_COUNT}): {len(VOCAB)} types")

print()
print("Top-10 most frequent tags:")
for tag, cnt in tag_freq.most_common(10):
    print(f"  {tag:<10} {cnt:>7}")

print()
print("Top-10 most frequent words:")
for word, cnt in word_freq.most_common(10):
    print(f"  {word:<20} {cnt:>7}")

In [ ]:
# ── Transition probability sanity check ─────────────────────────────────
print("═" * 50)
print(" TRANSITION SANITY CHECK")
print("═" * 50)

# Each row of log_transition[prev] must sum to ~1 in probability space
max_err = 0.0
for prev, row in log_transition.items():
    row_sum = sum(math.exp(lp) for lp in row.values())
    err     = abs(row_sum - 1.0)
    if err > max_err:
        max_err = err

print(f"Max |row_sum − 1| across all transition rows : {max_err:.2e}")
assert max_err < 1e-6, f"Transition rows do not sum to 1! max_err={max_err}"
print("PASS — all transition rows sum to 1.")

print()
print("Sample transitions from START_TAG:")
top_start = sorted(
    log_transition[START_TAG].items(),
    key=lambda x: x[1], reverse=True
)[:5]
for tag, lp in top_start:
    print(f"  P({tag:10} | START) = {math.exp(lp):.6f}")

In [ ]:
# ── Emission probability sanity check ───────────────────────────────────
print("═" * 50)
print(" EMISSION SANITY CHECK")
print("═" * 50)

max_err = 0.0
for tag, row in log_emission.items():
    row_sum = sum(math.exp(lp) for lp in row.values())
    err     = abs(row_sum - 1.0)
    if err > max_err:
        max_err = err

print(f"Max |row_sum − 1| across all emission rows : {max_err:.2e}")
# Note: smoothed emission rows sum to exactly 1 over the closed vocabulary
# (within floating-point tolerance)
assert max_err < 1e-4, f"Emission rows do not sum to 1! max_err={max_err}"
print("PASS — all emission rows sum to ≈1.")

print()
print("Top-5 most probable words for tag NOUN:")
if "NOUN" in log_emission:
    top_noun = sorted(log_emission["NOUN"].items(), key=lambda x: x[1], reverse=True)[:5]
    for word, lp in top_noun:
        print(f"  P({word:<20} | NOUN) = {math.exp(lp):.6f}")
else:
    print("  (NOUN not in tag set)")

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────
print("═" * 50)
print(" WEEK 1 COMPLETE")
print("═" * 50)
print(f"✓ Loaded  {len(train_sents)} training sentences")
print(f"✓ Trained HMM  |  tags={len(TAGS)}, vocab={len(VOCAB)}")
print(f"✓ Model saved  →  {MODEL_OUT}")
print()
print("Next step (Week 2): Implement the Viterbi algorithm for decoding.")